In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from ugdatalab.models.apogee.constants import LABEL_NAMES, LABEL_LATEX

import plotters

In [ ]:
# Load data and use the same train/CV split as the Cannon
spec_data = np.load("training_spectra.npz", allow_pickle=True)
flux = spec_data["flux"]
error = spec_data["error"]
labels = spec_data["labels"]

model_data = np.load("cannon_model.npz", allow_pickle=True)
train_idx = model_data["train_idx"]
cv_idx = model_data["cv_idx"]

flux_train, labels_train = flux[train_idx], labels[train_idx]
flux_cv, labels_cv = flux[cv_idx], labels[cv_idx]

print(f"Training: {len(train_idx)} stars")
print(f"CV: {len(cv_idx)} stars")
print(f"Pixels per spectrum: {flux.shape[1]}")

## Problem 16 — Neural Network Label Prediction

### Data preparation

We normalize labels to order unity (subtract mean, divide by std) using training-set statistics. Flux values are used as-is (already normalized to ~1). Pixels with `inf` errors are replaced with zero flux to avoid NaN propagation.

In [ ]:
# Normalize labels
label_mean = np.mean(labels_train, axis=0)
label_std = np.std(labels_train, axis=0)
labels_train_norm = (labels_train - label_mean) / label_std
labels_cv_norm = (labels_cv - label_mean) / label_std

# Replace inf/NaN flux with 0 (zero weight in practice)
flux_train_clean = np.nan_to_num(flux_train, nan=0.0, posinf=0.0, neginf=0.0)
flux_cv_clean = np.nan_to_num(flux_cv, nan=0.0, posinf=0.0, neginf=0.0)

# PyTorch datasets
device = torch.device("cpu")
train_dataset = TensorDataset(
    torch.tensor(flux_train_clean, dtype=torch.float32),
    torch.tensor(labels_train_norm, dtype=torch.float32),
)
cv_dataset = TensorDataset(
    torch.tensor(flux_cv_clean, dtype=torch.float32),
    torch.tensor(labels_cv_norm, dtype=torch.float32),
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                          generator=torch.Generator().manual_seed(42))
cv_loader = DataLoader(cv_dataset, batch_size=len(cv_dataset), shuffle=False)

n_pixels = flux_train_clean.shape[1]
n_labels = labels_train.shape[1]
print(f"Input dimension: {n_pixels}")
print(f"Output dimension: {n_labels}")

### Network definition

A simple MLP: `Linear(8575, 512) → ReLU → Linear(512, 256) → ReLU → Linear(256, 5)`.

In [ ]:
torch.manual_seed(42)

net = nn.Sequential(
    nn.Linear(n_pixels, 512),
    nn.ReLU(),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Linear(256, n_labels),
).to(device)

n_params = sum(p.numel() for p in net.parameters())
print(f"Network parameters: {n_params:,}")
print(net)

### Training loop

MSE loss on normalized labels. Adam optimizer with early stopping on validation loss (patience = 20 epochs).

In [ ]:
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
criterion = nn.MSELoss()

n_epochs = 200
patience = 20
best_val_loss = np.inf
epochs_without_improvement = 0
best_state = None

train_losses = []
val_losses = []

for epoch in range(1, n_epochs + 1):
    # Training
    net.train()
    epoch_loss = 0.0
    n_batches = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = net(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    train_losses.append(epoch_loss / n_batches)

    # Validation
    net.eval()
    with torch.no_grad():
        for X_val, y_val in cv_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)
            val_pred = net(X_val)
            val_loss = criterion(val_pred, y_val).item()
    val_losses.append(val_loss)

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        best_state = {k: v.clone() for k, v in net.state_dict().items()}
    else:
        epochs_without_improvement += 1

    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}: train={train_losses[-1]:.5f}, val={val_loss:.5f}")

    if epochs_without_improvement >= patience:
        print(f"Early stopping at epoch {epoch} (patience={patience})")
        break

# Restore best model
net.load_state_dict(best_state)
print(f"Best validation loss: {best_val_loss:.5f}")

### Loss curves

In [ ]:
ax = plotters.plot_nn_loss(train_losses, val_losses)
plt.show()

### CV evaluation

In [ ]:
# Predict on CV set and unnormalize
net.eval()
with torch.no_grad():
    X_cv_tensor = torch.tensor(flux_cv_clean, dtype=torch.float32).to(device)
    nn_pred_norm = net(X_cv_tensor).cpu().numpy()

nn_fitted_labels = nn_pred_norm * label_std + label_mean

axes = plotters.plot_nn_label_recovery(labels_cv, nn_fitted_labels, LABEL_LATEX)
plt.show()

### Cannon vs Neural Network comparison

In [ ]:
# Load Cannon CV results for comparison
cv_data = np.load("cv_results.npz", allow_pickle=True)
cannon_fitted = cv_data["fitted_labels"]

cannon_resid = cannon_fitted - labels_cv
nn_resid = nn_fitted_labels - labels_cv

comparison = pd.DataFrame({
    "Label": LABEL_NAMES,
    "Cannon bias": np.mean(cannon_resid, axis=0),
    "Cannon scatter": np.std(cannon_resid, axis=0),
    "NN bias": np.mean(nn_resid, axis=0),
    "NN scatter": np.std(nn_resid, axis=0),
})
comparison

### Discussion

**Cannon advantages**:
- Interpretable: gradient spectra reveal which spectral features drive each label; per-pixel scatter identifies model limitations.
- Explicit uncertainty model: intrinsic scatter $s_\lambda^2$ quantifies model inadequacy; can be propagated through MCMC for principled posteriors.
- Fast training: ~30–60 s for 8575 pixels (each pixel is an independent 1D optimization).
- Low risk of overfitting with 21 coefficients per pixel and ~1000 training stars.

**Neural network advantages**:
- Nonlinear: can capture complex relationships between spectra and labels that a 2nd-order polynomial cannot.
- No manual feature engineering: learns its own internal representation.
- Potentially lower scatter if the true mapping is highly nonlinear.

**Neural network disadvantages**:
- Black box: no gradient spectra or scatter diagnostics.
- No built-in uncertainty: requires additional techniques (dropout, ensembles, or normalizing flows) for uncertainty estimates.
- More parameters: higher risk of overfitting, especially with limited training data.
- Sensitivity to preprocessing: bad pixels filled with zeros affect the network differently than setting weight to zero in chi-squared.

### Save results

In [ ]:
np.savez_compressed(
    "nn_results.npz",
    nn_fitted_labels=nn_fitted_labels,
    true_labels=labels_cv,
)
print("Saved nn_results.npz")
print(f"  nn_fitted_labels: {nn_fitted_labels.shape}")
print(f"  true_labels: {labels_cv.shape}")